# Calculate transparent Renewal Risk Pulse

Every component is clipped to [0,1]. Weights sum to 100. No actuarial or predictive claim is made.


In [ ]:
import sys
from pathlib import Path
BASE = Path('/lakehouse/default/Files/underwriting_demo')
sys.path.insert(0, str(BASE / 'src'))
from insurance_demo import build_data, validate, column_type, AS_OF, ROOT
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, BooleanType
TABLE_PREFIX = 'demo_'

def save_rows(name, rows):
    types = {'bit': BooleanType(), 'bigint': LongType(), 'float': DoubleType()}
    schema = StructType([StructField(field, types.get(column_type(field, rows), StringType()), True) for field in rows[0]])
    frame = spark.createDataFrame(rows, schema)
    for field in rows[0]:
        datatype = column_type(field, rows)
        if datatype == 'date':
            frame = frame.withColumn(field, F.to_date(field))
        elif datatype == 'datetime2(6)':
            frame = frame.withColumn(field, F.to_timestamp(field))
    destination = TABLE_PREFIX + name
    if spark.catalog.tableExists(destination):
        properties = {row.key: row.value for row in spark.sql(f'SHOW TBLPROPERTIES {destination}').collect()}
        assert properties.get('delta.columnMapping.mode', 'none') == 'none', 'Use a new managed demo table without column mapping'
    frame.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(destination)
    assert spark.table(destination).count() == len(rows)
    return frame

from insurance_demo import WEIGHTS
features = spark.table(TABLE_PREFIX + 'UnderwritingFeatures')
score = sum(F.lit(weight) * F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col(component))) for component, weight in WEIGHTS.items())
pulses = features.select('AccountId', 'SnapshotDate', *WEIGHTS).withColumn('PulseId', F.concat(F.lit('PULSE'), F.col('AccountId'))).withColumn('Pulse', F.round(score, 2))
pulses.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(TABLE_PREFIX + 'RiskPulse')
assert pulses.filter(F.col('AccountId') == 'A001').first().Pulse == 77.0
display(pulses.orderBy(F.desc('Pulse')))
